# FCA change-type classifier - fine-tuned transformer (DistilBERT)
## Self-contained Colab run - Method C of the RQ2 three-way comparison

**Before you start (2 minutes):**
1. On your machine: `cd ~/fca_project && tar -czf data_text_labelling.tar.gz data/text data/labelling`
2. Upload `data_text_labelling.tar.gz` to Google Drive (root of MyDrive).
3. This notebook: Runtime -> Change runtime type -> **T4 GPU** -> Save.

**Then run every cell top to bottom.** On a T4 this takes ~20-40 minutes
(5 fine-tune folds on ~179 short documents).

**Pre-committed fallback (decided in advance, not after the fact):**
if any fold errors or training stalls, do NOT fight it - note the error,
tell me, and we ship the rule-based + zero-shot two-way comparison instead.
The fine-tune is presented as an attempted third method with honest analysis.

**Evaluation protocol (declared):** this method trains on labels, so it is
evaluated with stratified 5-fold cross-validation. Rule-based and zero-shot
use no labels at inference and are evaluated on all usable docs.


### 1) Mount Drive and unpack the data
Edit `TAR_PATH` only if you placed the archive elsewhere.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TAR_PATH = '/content/drive/MyDrive/data_text_labelling.tar.gz'
ROOT = '/content/fca'

import os, glob, subprocess
os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f'{ROOT}/data/text'):
    subprocess.run(['tar', '-xzf', TAR_PATH, '-C', ROOT], check=True)
    print('unpacked archive')
print('text files:', len(glob.glob(f'{ROOT}/data/text/*.txt')))
print('labels header:', open(f'{ROOT}/data/labelling/labels.csv').readline().strip())
assert len(glob.glob(f'{ROOT}/data/text/*.txt')) == 305, 'expected 305 text files - wrong archive?'
DATA = f'{ROOT}/data'


### 2) Install dependencies
One-time per runtime (~2 min). `torch` is bundled on Colab but pinned here to be explicit.


In [ ]:
!pip -q install torch transformers datasets scikit-learn accelerate
import torch, transformers, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - will be slow')


### 3) Load labels + text (same 179 usable docs as the other two methods)
Drops the single `unknown` row (cryptoasset_regime, JS-rendered) so the three
methods are compared on the identical document set.


In [ ]:
import csv, os
from collections import Counter

LABELS = ['new_rule', 'amendment', 'consultation', 'guidance', 'no_change']
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

meta = {}
with open(f'{DATA}/labelling/labels.csv') as f:
    for r in csv.DictReader(f):
        meta[r['stem']] = r

stems = [s for s, r in meta.items()
         if r['label'] in label2id and os.path.exists(f'{DATA}/text/{s}.txt')]
print(len(stems), 'usable docs')
print('class balance:', dict(Counter(meta[s]['label'] for s in stems)))


### 4) Tokenize (DistilBERT, max 512 tokens, text capped at 2000 chars)
Capping at 2000 chars mirrors exactly what the human labeller (and the
rule-based / zero-shot methods) read, keeping inputs comparable.


In [ ]:
from transformers import AutoTokenizer

MAX_TEXT_CHARS = 2000
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

texts = []
ys = []
for s in stems:
    t = open(f'{DATA}/text/{s}.txt', encoding='utf-8', errors='replace').read()[:MAX_TEXT_CHARS]
    texts.append(t)
    ys.append(label2id[meta[s]['label']])

enc = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors='pt')
print('input_ids shape:', tuple(enc['input_ids'].shape))


### 5) Stratified 5-fold fine-tuning (the honest CV protocol)
One DistilBERT per fold, trained on 4/5 of the data, predicting the held-out
1/5. Out-of-fold predictions are concatenated for a single unbiased score.


In [ ]:
import numpy as np, torch, time
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

oof = np.zeros((len(stems), len(LABELS)))
y_true_all = np.array(ys)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_metrics = []
start = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(enc['input_ids'], y_true_all), 1):
    print(f'--- fold {fold}/5 ---')
    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=len(LABELS)).to(device)

    train_ds = TensorDataset(enc['input_ids'][tr_idx], enc['attention_mask'][tr_idx], torch.tensor(ys)[tr_idx])
    val_ds   = TensorDataset(enc['input_ids'][va_idx], enc['attention_mask'][va_idx], torch.tensor(ys)[va_idx])
    train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)

    epochs = 3
    opt = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

    model.train()
    for ep in range(epochs):
        for input_ids, attn, lbl in train_dl:
            input_ids, attn, lbl = input_ids.to(device), attn.to(device), lbl.to(device)
            loss = model(input_ids=input_ids, attention_mask=attn, labels=lbl).loss
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()

    model.eval()
    with torch.no_grad():
        for input_ids, attn, lbl in DataLoader(val_ds, batch_size=16):
            input_ids, attn = input_ids.to(device), attn.to(device)
            logits = model(input_ids=input_ids, attention_mask=attn).logits
            oof[va_idx] = torch.softmax(logits, dim=1).cpu().numpy()

    yv = y_true_all[va_idx]; yp = oof[va_idx].argmax(1)
    fa = accuracy_score(yv, yp); fm = f1_score(yv, yp, average='macro')
    fold_metrics.append({'fold': fold, 'accuracy': round(float(fa), 4), 'macro_f1': round(float(fm), 4)})
    print(f'  fold {fold}: accuracy={fa:.3f}  macro-F1={fm:.3f}')

import json
print('total time: %.1f min' % ((time.time() - start) / 60))
print('per-fold:', json.dumps(fold_metrics))


### 6) Write out-of-fold predictions (the method-C CSV) + metrics
`fine_tuned.csv` holds one prediction per doc from a model that never saw it
during training (out-of-fold). This is what gets compared in the RQ2 report.


In [ ]:
import csv, json

preds = [id2label[int(oof[i].argmax())] for i in range(len(stems))]
confs = [float(oof[i].max()) for i in range(len(stems))]

os.makedirs(f'{DATA}/predictions', exist_ok=True)
with open(f'{DATA}/predictions/fine_tuned.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['stem', 'label', 'confidence'])
    for s, p, c in zip(stems, preds, confs):
        w.writerow([s, p, f'{c:.4f}'])

y_pred_all = np.array([id2label[y] for y in preds])
y_true_str = np.array([id2label[y] for y in ys])
overall = {
    'n': len(stems),
    'accuracy': round(float(accuracy_score(y_true_str, y_pred_all)), 4),
    'macro_f1': round(float(f1_score(y_true_str, y_pred_all, average='macro')), 4),
    'per_fold': fold_metrics,
}
with open(f'{DATA}/predictions/fine_tuned_metrics.json', 'w') as f:
    json.dump(overall, f, indent=2)
print(json.dumps(overall, indent=2))
print('wrote', f'{DATA}/predictions/fine_tuned.csv')


### 7) Package results + download
Saves a zip to Drive (and triggers a browser download) containing the two
files you need to copy back into `~/fca_project/data/predictions/`:
`fine_tuned.csv` and `fine_tuned_metrics.json`.


In [ ]:
import zipfile

zpath = '/content/fine_tuned_results.zip'
with zipfile.ZipFile(zpath, 'w') as z:
    z.write(f'{DATA}/predictions/fine_tuned.csv', 'fine_tuned.csv')
    z.write(f'{DATA}/predictions/fine_tuned_metrics.json', 'fine_tuned_metrics.json')
!cp {zpath} /content/drive/MyDrive/fine_tuned_results.zip
print('saved to Drive: /content/drive/MyDrive/fine_tuned_results.zip')

from google.colab import files
files.download(zpath)
